# Tugas Besar 2 IF3170: Prediksi Pembatalan Pemesanan Penginapan

# Initialization

## Import Libraries

In [ ]:
import os
import pickle
import random
import time

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Kode From Scratch

In [ ]:
# simpan model ke file pickle
def save_model(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(model, f)

# load lagi model yang disimpan pakai save_model
def load_model(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# confusion matrix biner, bentuknya [[TN, FP], [FN, TP]]
def confusion_matrix(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1))); tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1))); fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return np.array([[tn, fp], [fn, tp]])

# F1 buat class 0 dan class 1
def f1_per_class(y_true, y_pred):
    (tn, fp), (fn, tp) = confusion_matrix(y_true, y_pred)
    return np.array([2 * tn / (2 * tn + fn + fp), 2 * tp / (2 * tp + fp + fn)])

In [ ]:
TARGET = 'is_canceled'
ID_COL = 'id'
CAT_COLS = ['meal_package', 'room_category', 'booking_channel']
RAW = ('num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'meal_package', 'needs_parking',
       'room_category', 'days_before_arrival', 'arrival_year', 'arrival_month', 'arrival_day', 'booking_channel',
       'is_returning_guest', 'prior_cancellations', 'prior_completed_bookings', 'nightly_rate_idr', 'special_requests')
ENGINEERED = ('total_nights', 'total_guests', 'is_free', 'rate_per_guest', 'arrival_weekday',
              'arrival_dayofyear', 'arrival_index', 'booking_index', 'doy_sin', 'doy_cos')
NEW = ('weekend_ratio', 'total_cost', 'has_children', 'is_weekend_arrival', 'month_sin', 'month_cos')
POOL = RAW + ENGINEERED + NEW

FEATURE_SETS = {
    'raw': RAW,
    'raw+fe': RAW + ENGINEERED,
    'v2': ('days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
           'booking_channel', 'arrival_dayofyear', 'total_nights', 'needs_parking'),
    'combo1347': ('days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
                  'booking_channel', 'doy_sin', 'doy_cos', 'total_nights', 'needs_parking', 'booking_index'),
}

CLIP_COLS = ['num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival',
             'nightly_rate_idr', 'prior_cancellations', 'prior_completed_bookings', 'special_requests']
LOG_COLS = {'days_before_arrival', 'nightly_rate_idr', 'prior_cancellations', 'prior_completed_bookings',
            'weekend_nights', 'weekday_nights', 'total_nights', 'rate_per_guest', 'total_cost'}
ORD_MAPS = {'meal_package': {'Room Only': 0, 'Breakfast': 1, 'Half Board': 2, 'Full Board': 3}}


# Data train setelah cleaning baris: label bertentangan lalu dedupe.
def get_train(cfg):
    df = train.copy()
    feat_cols = [c for c in df.columns if c not in (ID_COL, TARGET)]
    if cfg['conflict'] != 'keep':
        key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)
        mean = df.groupby(key)[TARGET].transform('mean')
        if cfg['conflict'] == 'drop':
            df = df[(mean == 0) | (mean == 1)]
        else:
            df = df[mean != 0.5].copy()
            df[TARGET] = (mean[mean != 0.5] > 0.5).astype(int)
    if cfg['dedupe']:
        df = df.drop_duplicates(subset=feat_cols + [TARGET])
    return df.reset_index(drop=True)


# Tandai nilai janggal sebagai missing.
def mark_invalid(df, cfg):
    df = df.copy()
    if cfg['invalid']:
        df.loc[df['num_adults'] == 0, 'num_adults'] = np.nan
    if cfg['invalid_extra']:
        zero = (df['weekend_nights'] + df['weekday_nights']) == 0
        df.loc[zero, ['weekend_nights', 'weekday_nights']] = np.nan
        df.loc[df['num_children'] >= 9, 'num_children'] = np.nan
        df.loc[(df['nightly_rate_idr'] == 0) & (df['booking_channel'] != 'Complimentary'), 'nightly_rate_idr'] = np.nan
    return df


# Batas clipping dari data train. Kolom yang batasnya sama tidak dipotong.
def clip_bounds(df, how):
    x = df[CLIP_COLS]
    if how == 'iqr':
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    else:
        a = {'p99': 0.01, 'p995': 0.005}[how]
        lo, hi = x.quantile(a), x.quantile(1 - a)
    same = lo >= hi
    return lo.where(~same, -np.inf), hi.where(~same, np.inf)


# Semua fitur turunan. Kolom yang dipakai ditentukan oleh cfg['features'].
def engineer(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    df.loc[df['nightly_rate_idr'].isna(), 'is_free'] = np.nan
    df['rate_per_guest'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)
    arrival = pd.to_datetime(dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
                             errors='coerce')
    df['arrival_weekday'] = arrival.dt.day_of_week.astype(float)
    df['arrival_dayofyear'] = arrival.dt.day_of_year.astype(float)
    df['arrival_index'] = (arrival - pd.Timestamp('2023-01-01')).dt.days.astype(float)
    df['booking_index'] = df['arrival_index'] - df['days_before_arrival']
    df['doy_sin'] = np.sin(2 * np.pi * df['arrival_dayofyear'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['arrival_dayofyear'] / 365.25)
    df['weekend_ratio'] = df['weekend_nights'] / df['total_nights'].replace(0, np.nan)
    df['total_cost'] = df['nightly_rate_idr'] * df['total_nights']
    df['has_children'] = (df['num_children'] > 0).astype(float).where(df['num_children'].notna())
    df['is_weekend_arrival'] = (df['arrival_weekday'] >= 5).astype(float).where(df['arrival_weekday'].notna())
    df['month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)
    return df


# Preprocessing bersama KNN dan DTL. Semua statistik dipelajari dari fit_df saja.
# Hasilnya frame fit dan list frame lain yang sudah diproses.
def common_frames(cfg, fit_df, *frames):
    raw = [mark_invalid(d, cfg) for d in (fit_df,) + frames]
    if cfg['clip']:
        lo, hi = clip_bounds(raw[0], cfg['clip'])
        for d in raw:
            d[CLIP_COLS] = d[CLIP_COLS].clip(lower=lo, upper=hi, axis=1)
    eng = [engineer(d) for d in raw]
    F = eng[0]
    if cfg['rare']:
        for c in CAT_COLS:
            counts = F[c].value_counts()
            keep = set(counts[counts >= cfg['rare']].index)
            for d in eng:
                d.loc[d[c].notna() & ~d[c].isin(keep), c] = 'Other'
    num = [c for c in F.columns if c not in CAT_COLS and c not in (ID_COL, TARGET)]
    if cfg['miss_ind']:
        for c in num:
            if F[c].isna().any():
                for d in eng:
                    d['miss_' + c] = d[c].isna().astype(float)
    if cfg['impute']:
        stat = {'median': F[num].median(), 'mean': F[num].mean(), 'mode': F[num].mode().iloc[0]}[cfg['impute']]
        fill = {**stat.to_dict(), **{c: F[c].mode()[0] for c in CAT_COLS}}
        eng = [d.fillna(fill) for d in eng]
    return eng[0], eng[1:]


# Kode angka untuk kategori: ordinal atau frequency, dipelajari dari frame fit F.
def cat_codes(F, d, c, how):
    if how == 'freq':
        return d[c].map(F[c].value_counts(normalize=True)).astype(float).fillna(0.0)
    mapping = ORD_MAPS.get(c) or {v: i for i, v in enumerate(sorted(F[c].dropna().unique()))}
    return d[c].map(mapping).astype(float)

In [ ]:
# Macro F1 biner: rata-rata F1 kelas 0 dan kelas 1.
def macro_f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1)); tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1)); fn = np.sum((y_true == 1) & (y_pred == 0))
    f1_pos = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0
    f1_neg = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp else 0.0
    return float((f1_pos + f1_neg) / 2)


# Threshold dengan Macro F1 tertinggi pada prediksi OOF.
def best_threshold(y_true, proba):
    grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    return float(grid[int(np.argmax([macro_f1(y_true, proba >= t) for t in grid]))])


# Stratified k-fold manual, hasilnya list (train_idx, val_idx).
def stratified_kfold(y, n_splits=5, seed=42):
    rng = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        fold_of[idx] = np.arange(len(idx)) % n_splits
    return [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(n_splits)]

In [ ]:
# Matriks KNN untuk fit_df dan frame lain.
def knn_prepare(cfg, fit_df, *frames):
    F, outs = common_frames(cfg, fit_df, *frames)
    feats = list(cfg['features'])
    num = [f for f in feats if f not in CAT_COLS]
    cat = [f for f in feats if f in CAT_COLS]
    ind = [f'miss_{f}' for f in num if f'miss_{f}' in F.columns]
    coded = cat if cfg['cat_enc'] != 'onehot' else []
    w = dict(cfg['fw'])

    def numeric(d):
        X = d[num].astype(float).copy()
        for c in coded:
            X[c] = cat_codes(F, d, c, cfg['cat_enc'])
        if cfg['log']:
            for c in X.columns:
                if c in LOG_COLS:
                    X[c] = np.log1p(X[c].clip(lower=0))
        return X

    XF = numeric(F)
    med = XF.median()
    XF = XF.fillna(med)
    s = cfg['scaler']
    if s == 'zscore':
        center, scale = XF.mean(), XF.std()
    elif s == 'minmax':
        center, scale = XF.min(), XF.max() - XF.min()
    elif s == 'robust':
        center, scale = XF.median(), XF.quantile(0.75) - XF.quantile(0.25)
        scale = scale.where(scale > 0, XF.std())
    else:
        sorted_vals = {c: np.sort(XF[c].to_numpy()) for c in XF.columns}
    result = []
    for d in outs:
        X = numeric(d).fillna(med)
        if s == 'rank':
            Z = pd.DataFrame({c: ((np.searchsorted(sorted_vals[c], X[c], 'left') + np.searchsorted(sorted_vals[c], X[c], 'right'))
                                  / (2 * len(sorted_vals[c])) - 0.5) * np.sqrt(12) for c in X.columns}, index=X.index)
        else:
            Z = (X - center) / scale.replace(0, 1)
        parts = [Z[c].to_numpy()[:, None] * w.get(c, 1.0) for c in Z.columns]
        parts += [d[c].to_numpy()[:, None] * cfg['onehot'] for c in ind]
        if cfg['cat_enc'] == 'onehot':
            for c in cat:
                v = d[c].fillna(F[c].mode()[0]).to_numpy()
                levels = sorted(F[c].dropna().unique())
                parts.append(np.stack([(v == lv) for lv in levels], axis=1) * cfg['onehot'] * w.get(c, 1.0))
        result.append(np.hstack(parts).astype(np.float32))
    return result


# k tetangga terdekat dengan jarak Minkowski, terurut dari yang terdekat.
def kneighbors(X, Q, k, p, batch=256):
    ds, ids = [], []
    for s in range(0, len(Q), batch):
        q = Q[s:s + batch]
        D = np.zeros((len(q), len(X)), np.float32)
        for j in range(X.shape[1]):
            D += np.abs(q[:, j, None] - X[None, :, j]) ** p
        idx = np.argpartition(D, k - 1, axis=1)[:, :k]
        d = np.take_along_axis(D, idx, 1)
        o = np.argsort(d, axis=1, kind='stable')
        ids.append(np.take_along_axis(idx, o, 1)); ds.append(np.take_along_axis(d, o, 1) ** (1 / p))
    return np.vstack(ds), np.vstack(ids)


# Edited Nearest Neighbors: baris yang labelnya beda dari mayoritas k tetangganya ditandai noise.
# Hasilnya (mask baris yang disimpan, label baru).
def enn(X, y, k, p, mode):
    _, idx = kneighbors(X, X, k + 1, p)
    is_self = idx == np.arange(len(X))[:, None]
    keep = ~is_self
    keep[~is_self.any(1), -1] = False
    nb = idx[keep].reshape(len(X), k)
    share = y[nb].mean(1)
    noisy = ((share > 0.5) & (y == 0)) | ((share < 0.5) & (y == 1))
    if mode == 'drop':
        return ~noisy, y
    return np.ones(len(y), bool), np.where(noisy, 1 - y, y)


# Probabilitas kelas 1 dari vote tetangga dengan bobot sesuai cfg['weight'].
def knn_fit_predict(cfg, fit_df, pred_df):
    X, Q = knn_prepare(cfg, fit_df, fit_df, pred_df)
    y = fit_df[TARGET].to_numpy().astype(int)
    if cfg['enn_k']:
        mask, y = enn(X, y, cfg['enn_k'], cfg['p'], cfg['enn_mode'])
        X, y = X[mask], y[mask]
    d, idx = kneighbors(X, Q, cfg['k'], cfg['p'])
    if cfg['weight'] == 'uniform':
        w = np.ones_like(d)
    elif cfg['weight'] == 'inv':
        w = 1 / (d + 1e-6)
    elif cfg['weight'] == 'inv2':
        w = 1 / (d ** 2 + 1e-6)
    else:
        w = np.exp(-2 * (d / (d[:, -1:] + 1e-6)) ** 2)
    return (w * y[idx]).sum(1) / w.sum(1)

In [ ]:
# kolom yang harus sama persis supaya dua booking dianggap kembar
TWIN_KEY = ['weekend_nights', 'weekday_nights', 'booking_channel', 'is_returning_guest', 'prior_cancellations',
            'prior_completed_bookings']
# kolom yang harus sama kalau dua-duanya terisi (missing dianggap cocok)
TWIN_SOFT = ['num_adults', 'num_children', 'needs_parking', 'meal_package', 'room_category', 'special_requests']

def add_date(df):
    df = df.copy()
    df['date'] = pd.to_datetime(dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
                                errors='coerce')
    return df

# cari booking kembar di B untuk tiap row A: kolom TWIN_KEY sama, TWIN_SOFT sama,
# lead time dan tanggal datang beda <= lead/day hari, harga beda <= price (relatif)
# hasilnya (proporsi kembar yang batal, jumlah kembar)
def twin_vote(A, B, yB, lead, day, price):
    A, B = add_date(A).reset_index(drop=True), add_date(B).reset_index(drop=True)
    share = np.full(len(A), np.nan)
    count = np.zeros(len(A), int)
    groups_b = B.groupby(TWIN_KEY, dropna=False).indices
    for key, ia in A.groupby(TWIN_KEY, dropna=False).indices.items():
        ib = groups_b.get(key)
        if ib is None:
            continue
        a, b = A.iloc[ia], B.iloc[ib]
        ok = np.ones((len(ia), len(ib)), bool)
        for c in TWIN_SOFT:
            x, z = a[c].to_numpy()[:, None], b[c].to_numpy()[None, :]
            ok &= (x == z) | pd.isna(x) | pd.isna(z)
        x, z = a['days_before_arrival'].to_numpy(float)[:, None], b['days_before_arrival'].to_numpy(float)[None, :]
        ok &= (np.abs(x - z) <= lead) | np.isnan(x) | np.isnan(z)
        dd = np.abs((a['date'].to_numpy()[:, None] - b['date'].to_numpy()[None, :]) / np.timedelta64(1, 'D')).astype(float)
        ok &= (dd <= day) | np.isnan(dd)
        x, z = a['nightly_rate_idr'].to_numpy(float)[:, None], b['nightly_rate_idr'].to_numpy(float)[None, :]
        ok &= (np.abs(x - z) <= price * np.maximum(z, 1)) | np.isnan(x) | np.isnan(z)
        n = ok.sum(1)
        share[ia] = np.where(n > 0, (ok * yB[ib][None, :]).sum(1) / np.maximum(n, 1), np.nan)
        count[ia] = n
    return share, count

# KNN dengan tetangga kembar: kalau ada minimal min_count kembar,
# probabilitas = (1 - alpha) * vote k neighbor + alpha * vote kembar
def knn_twin_proba(cfg, twin, fit_df, pred_df, knn_proba=None):
    p = knn_fit_predict(cfg, fit_df, pred_df) if knn_proba is None else knn_proba.copy()
    share, count = twin_vote(pred_df, fit_df, fit_df[TARGET].to_numpy().astype(int), twin['lead'], twin['day'], twin['price'])
    use = count >= twin['min_count']
    p[use] = (1 - twin['alpha']) * p[use] + twin['alpha'] * share[use]
    return p

## Settings

In [ ]:
KNN_BASE = dict(dedupe=False, conflict='keep', invalid=False, invalid_extra=False, clip=None, impute='median', miss_ind=False, rare=0, cat_enc='onehot', onehot=1.0, log=False, scaler='zscore', features=FEATURE_SETS['raw'], fw=(), k=5, p=2.0, weight='uniform', enn_k=0, enn_mode='drop', threshold='0.5')

class Settings:
    SEED = 42
    SEEDS = [1, 2, 3]
    CHECK_SEEDS = [4, 5, 42]
    CONFIG = {**KNN_BASE, **dict(dedupe=True, invalid=True, clip='p995', onehot=0.75, scaler='robust',
                  features=('num_adults', 'weekend_nights', 'weekday_nights', 'meal_package', 'needs_parking',
                            'room_category', 'days_before_arrival', 'arrival_day', 'booking_channel', 'is_returning_guest',
                            'prior_cancellations', 'nightly_rate_idr', 'special_requests', 'booking_index', 'month_cos'),
                  fw=(('days_before_arrival', 2.0), ('needs_parking', 0.5), ('num_adults', 0.25), ('special_requests', 2.0),
                      ('weekday_nights', 0.5)),
                  k=17, p=1.5, weight='inv2'), 'threshold': '0.5'}
    EXP_NAME = 'knn_robust_clip_minkowski_weighted_twin'
    MODEL_PATH = f'{EXP_NAME}.pkl'
    SUB_PATH = f'submission_{EXP_NAME}.csv'

CFG = Settings()
{k: v for k, v in CFG.CONFIG.items() if KNN_BASE[k] != v}

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

In [ ]:
train = pd.read_csv('train.csv', dtype={'id': str})
test = pd.read_csv('test.csv', dtype={'id': str})
print('train', train.shape, ' test', test.shape)

# Data Preparation

## Cleaning

In [ ]:
tr = get_train(CFG.CONFIG)
y = tr[TARGET].to_numpy()
print('train setelah cleaning', tr.shape)

## Feature Engineering

In [ ]:
engineer(tr)[[f for f in CFG.CONFIG['features'] if f not in CAT_COLS]].describe().T.round(2)

## Preprocessor KNN

In [ ]:
X_demo, = knn_prepare(CFG.CONFIG, tr, test.head())
print('jumlah kolom setelah preprocessing', X_demo.shape[1])
X_demo[:3].round(3)

## Booking Kembar

In [ ]:
# EDA: berapa banyak booking punya kembar di train fold lain, dan seberapa sering label mayoritas kembar benar
rows = []
for tol in [(2, 2, 0.02), (5, 5, 0.05), (10, 10, 0.1)]:
    share = np.full(len(y), np.nan); count = np.zeros(len(y), int)
    for a, b in stratified_kfold(y, 5, CFG.SEED):
        s_, c_ = twin_vote(tr.iloc[b], tr.iloc[a], y[a], *tol)
        share[b], count[b] = s_, c_
    has = count > 0
    rows.append({'toleransi (hari, hari, harga)': tol, 'punya kembar': has.mean(),
                 'label mayoritas kembar benar': ((share[has] >= 0.5) == y[has]).mean(),
                 'rata-rata jumlah kembar': count[has].mean()})
_, test_count = twin_vote(test, tr, y, 5, 5, 0.05)
print('test yang punya kembar di train (toleransi 5, 5, 5%):', round((test_count > 0).mean(), 3))
pd.DataFrame(rows).round(3)

<!-- alasan-kembar -->

# Modeling dan Validation

## 4.1 Skema Validasi

In [ ]:
# OOF 5 fold di beberapa seed, threshold dari gabungan OOF kalau setting threshold = 'oof'
def oof_proba(cfg, seed):
    oof = np.zeros(len(y))
    for a, b in stratified_kfold(y, 5, seed):
        oof[b] = knn_fit_predict(cfg, tr.iloc[a], tr.iloc[b])
    return oof

## 4.2 KNN dengan Tetangga Kembar

In [ ]:
# OOF KNN saja di seed 42, dipakai ulang buat semua varian kembar
oof_knn = oof_proba(CFG.CONFIG, CFG.SEED)
base_th = best_threshold(y, oof_knn)
print('KNN saja: OOF', round(macro_f1(y, oof_knn >= base_th), 4), ' threshold', base_th)

# coba toleransi, bobot alpha, dan minimal jumlah kembar, pilih yang OOF-nya paling tinggi
rows = []
for tol in [(2, 2, 0.02), (5, 5, 0.05), (10, 10, 0.1)]:
    share = np.full(len(y), np.nan); count = np.zeros(len(y), int)
    for a, b in stratified_kfold(y, 5, CFG.SEED):
        s_, c_ = twin_vote(tr.iloc[b], tr.iloc[a], y[a], *tol)
        share[b], count[b] = s_, c_
    for alpha in [0.3, 0.5, 0.7]:
        for min_count in [1, 2]:
            p = oof_knn.copy(); use = count >= min_count
            p[use] = (1 - alpha) * p[use] + alpha * share[use]
            th = best_threshold(y, p)
            rows.append({'lead': tol[0], 'day': tol[1], 'price': tol[2], 'alpha': alpha, 'min_count': min_count,
                         'threshold': th, 'OOF': macro_f1(y, p >= th)})
grid = pd.DataFrame(rows).sort_values('OOF', ascending=False).reset_index(drop=True)
best = grid.iloc[0]
TWIN = {'lead': int(best['lead']), 'day': int(best['day']), 'price': float(best['price']),
        'alpha': float(best['alpha']), 'min_count': int(best['min_count'])}
THRESHOLD = float(best['threshold'])
print('kembar terpilih', TWIN, ' threshold', THRESHOLD)
grid.head(10).round(4)

## 4.3 Cek dengan Pembagian Fold Lain

In [ ]:
rows = []
for seed in CFG.CHECK_SEEDS:
    if seed == CFG.SEED:
        continue
    o = oof_proba(CFG.CONFIG, seed)
    share = np.full(len(y), np.nan); count = np.zeros(len(y), int)
    for a, b in stratified_kfold(y, 5, seed):
        s_, c_ = twin_vote(tr.iloc[b], tr.iloc[a], y[a], TWIN['lead'], TWIN['day'], TWIN['price'])
        share[b], count[b] = s_, c_
    p = o.copy(); use = count >= TWIN['min_count']
    p[use] = (1 - TWIN['alpha']) * p[use] + TWIN['alpha'] * share[use]
    rows.append({'seed': seed, 'KNN saja': macro_f1(y, o >= best_threshold(y, o)),
                 'KNN + kembar': macro_f1(y, p >= best_threshold(y, p))})
check = pd.DataFrame(rows).set_index('seed')
check['selisih'] = check['KNN + kembar'] - check['KNN saja']
print('KNN + kembar unggul di', int((check['selisih'] > 0).sum()), 'dari', len(check), 'seed')
check.round(4)

# Error Analysis

## Confusion Matrix

In [ ]:
share = np.full(len(y), np.nan); count = np.zeros(len(y), int)
for a, b in stratified_kfold(y, 5, CFG.SEED):
    s_, c_ = twin_vote(tr.iloc[b], tr.iloc[a], y[a], TWIN['lead'], TWIN['day'], TWIN['price'])
    share[b], count[b] = s_, c_
oof = oof_knn.copy(); use = count >= TWIN['min_count']
oof[use] = (1 - TWIN['alpha']) * oof[use] + TWIN['alpha'] * share[use]
pred = (oof >= THRESHOLD).astype(int)
print('confusion matrix [[TN, FP], [FN, TP]]')
print(confusion_matrix(y, pred))
print('F1 kelas 0, kelas 1', f1_per_class(y, pred).round(4))

## Kesalahan per Kelompok

In [ ]:
def error_by(column, bins=None):
    values = tr[column] if bins is None else pd.qcut(tr[column], bins, duplicates='drop')
    frame = pd.DataFrame({column: values, 'error': pred != y})
    return frame.groupby(column, observed=True, dropna=False)['error'].agg(['mean', 'size']).round(3)

error_by('booking_channel')

In [ ]:
error_by('days_before_arrival', bins=5)

In [ ]:
error_by('special_requests')

<!-- improvement -->

# Submission

## 6.1 Training Final

In [ ]:
# KNN itu lazy learner, jadi "model" = konfigurasi + data train + aturan kembar + threshold
model = {'config': CFG.CONFIG, 'twin': TWIN, 'train': tr, 'threshold': THRESHOLD}
print('config', {k: v for k, v in CFG.CONFIG.items() if KNN_BASE[k] != v})
print('twin', TWIN, ' threshold', THRESHOLD)

## 6.2 Save dan Load Model

In [ ]:
save_model(model, CFG.MODEL_PATH)
bundle = load_model(CFG.MODEL_PATH)
print('model tersimpan di', CFG.MODEL_PATH)

## 6.3 Prediksi dan File Submission

In [ ]:
proba = knn_twin_proba(bundle['config'], bundle['twin'], bundle['train'], test)
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: (proba >= bundle['threshold']).astype(int)})
submission.to_csv(CFG.SUB_PATH, index=False)
submission[TARGET].value_counts(normalize=True).round(4)